In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Dados da Query Q1.5
csv_q1_5 = """project,blockchain,transaction_type,n_eventos,n_depositors,primeiro,ultimo
aave,polygon,deposit,20793916,318808,2021-03-31 10:39:27,2026-07-25 14:28:28
aave,polygon,withdraw,12692027,263450,2021-03-31 12:44:53,2026-07-25 14:08:25
aave,base,deposit,5657447,324108,2023-08-22 14:48:51,2026-07-25 14:32:19
aave,base,withdraw,4504745,260562,2023-08-22 14:53:39,2026-07-25 14:32:29
aave,arbitrum,deposit,3631883,278456,2022-03-16 16:00:45,2026-07-25 14:32:20
aave,arbitrum,withdraw,2960198,225476,2022-03-16 16:42:02,2026-07-25 14:32:09
aave,ethereum,deposit,2139324,260350,2019-12-16 22:20:30,2026-07-25 14:29:35
aave,ethereum,withdraw,1780821,219069,2020-01-06 23:11:52,2026-07-25 14:29:47
aave,polygon,deposit_liquidation,168479,42334,2021-04-14 15:07:46,2026-07-25 13:27:54
aave,ethereum,deposit_liquidation,87137,25519,2020-01-13 15:51:31,2026-07-25 12:09:47
aave,arbitrum,repay_with_atokens,59832,5590,2022-03-16 22:53:20,2026-07-25 14:26:02
aave,arbitrum,deposit_liquidation,51816,26718,2022-04-30 04:04:26,2026-07-24 13:08:39
aave,base,deposit_liquidation,45292,31909,2023-09-01 23:54:55,2026-07-24 13:06:11
aave,polygon,repay_with_atokens,40898,4842,2022-03-16 19:45:12,2026-07-24 22:15:07
aave,base,repay_with_atokens,15582,4627,2023-08-23 10:26:17,2026-07-25 07:26:31
aave,ethereum,repay_with_atokens,9412,3857,2023-01-27 14:37:23,2026-07-25 08:43:23
compound,base,supply,883081,60267,2023-08-12 17:42:53,2026-07-25 14:44:37
compound,ethereum,deposit,690404,344848,2019-05-07 01:41:22,2025-12-08 16:12:11
compound,base,withdraw,658808,175251,2023-08-13 00:08:19,2026-07-25 14:46:05
compound,ethereum,withdraw,419557,109942,2019-05-08 04:24:29,2026-07-25 14:29:59
compound,polygon,supply,324114,31803,2023-03-07 10:52:56,2026-07-24 16:48:45
compound,polygon,withdraw,313320,16797,2023-03-07 11:04:12,2026-07-25 14:13:04
compound,arbitrum,supply,144133,44339,2023-05-14 22:37:48,2026-07-25 14:23:50
compound,ethereum,supply,112994,15205,2022-08-26 01:00:26,2026-07-25 14:32:35
compound,arbitrum,withdraw,94411,28167,2023-05-16 03:04:58,2026-07-25 14:21:18
compound,base,supply_liquidation,3928,3712,2023-08-17 21:42:15,2026-07-20 17:51:23
compound,ethereum,supply_liquidation,2528,1370,2022-08-27 02:37:31,2026-07-08 07:16:35
compound,arbitrum,supply_liquidation,1930,1488,2023-06-05 15:58:53,2026-07-02 13:19:33
compound,polygon,supply_liquidation,515,330,2023-03-09 21:23:16,2026-06-27 16:14:52"""

df = pd.read_csv(io.StringIO(csv_q1_5))

# 2. Mapeamento
def map_tx_type(tx):
    if tx in ['deposit', 'supply']:
        return 'Deposit / Supply'
    elif tx == 'withdraw':
        return 'Withdrawal'
    elif 'liquidation' in tx:
        return 'Collateral Liquidation'
    else:
        return 'Other (Repay w/ Tokens)'

df['type_clean'] = df['transaction_type'].apply(map_tx_type)

# 3. Pivotar e calcular totais
df_grouped = df.groupby(['project', 'blockchain', 'type_clean'])['n_eventos'].sum().reset_index()
df_pivot = df_grouped.pivot(index=['project', 'blockchain'], columns='type_clean', values='n_eventos').fillna(0)

# Reordenar colunas
cols_order = ['Deposit / Supply', 'Withdrawal', 'Collateral Liquidation', 'Other (Repay w/ Tokens)']
cols_present = [c for c in cols_order if c in df_pivot.columns]
df_pivot = df_pivot[cols_present]

df_pivot['Total'] = df_pivot.sum(axis=1)

# Normalizar para porcentagem (100%)
df_perc = df_pivot[cols_present].div(df_pivot['Total'], axis=0) * 100

# Ordenar por volume total para a ordenação vertical fazer sentido
df_perc['Total'] = df_pivot['Total']
df_perc = df_perc.sort_values(by='Total', ascending=True).reset_index()

df_perc['label'] = df_perc['project'].str.capitalize() + ' (' + df_perc['blockchain'].str.capitalize() + ')'

# 4. Plotagem
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(11, 7))

colors = {
    'Deposit / Supply': '#2b5c8f',       # Azul escuro
    'Withdrawal': '#d95f02',             # Laranja/Vermelho
    'Collateral Liquidation': '#e7298a', # Rosa/Roxo para destacar liquidação
    'Other (Repay w/ Tokens)': '#7570b3' # Roxo
}

bottom = [0] * len(df_perc)
for col in cols_present:
    ax.barh(
        df_perc['label'],
        df_perc[col],
        left=bottom,
        label=col,
        color=colors[col],
        alpha=0.88,
        height=0.6
    )
    bottom = [b + v for b, v in zip(bottom, df_perc[col])]

# 5. Adicionar o Total Absoluto ao lado de cada barra
for idx, row in df_perc.iterrows():
    total_val = row['Total']
    # Formatação limpa em Milhões (M) ou Milhares (k)
    if total_val >= 1e6:
        total_str = f" {total_val/1e6:.2f}M total"
    else:
        total_str = f" {total_val/1e3:.0f}k total"

    ax.text(
        101, idx, total_str,
        va='center', ha='left',
        fontsize=8.5, fontweight='bold', color='#333333'
    )

# Formatação final
ax.set_xlim(0, 122) # Espaço para o texto do total à direita
ax.set_title('Q1.5 — Event Type Composition and Total Volume in lending.supply (100% Stacked)', fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('Proportion of Events (%)', fontsize=10, fontweight='bold')
ax.set_ylabel('')

# Estilizar o Eixo X para mostrar porcentagem
ax.xaxis.set_major_formatter('{x:.0f}%')

ax.legend(title='Event Type', loc='upper left', bbox_to_anchor=(0.0, -0.12), ncol=4, frameon=True, facecolor='white')

plt.tight_layout()

# Salvar
plt.savefig('Q1_5_cobertura_supply_100perc.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Q1_5_cobertura_supply_100perc.png', dpi=300, bbox_inches='tight')
plt.show()